In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px
import plotly.graph_objects as go

#Data Loading And Cleaning

df = pd.read_csv("dataset.csv")
df_clean = df.drop_duplicates(subset=["track_name","artists"])
df_clean = df.dropna(subset=["artists","album_name","track_name"])

target_genres = ['classical', 'black-metal', 'dance', 'acoustic', 'hip-hop','indian','rock','pop-film']
df_filtered = df_clean[df_clean['track_genre'].isin(target_genres)].copy()

print(f"Dataset shape: {df_filtered.shape}")
print(f"Genres included: {df_filtered['track_genre'].unique()}")

# Feature Extraction And Scaling
audio_features = ["danceability", "energy",  "loudness", "speechiness",
                  "acousticness", "instrumentalness", "liveness", "valence", "tempo"]

X = df_filtered[audio_features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["Genre"] = df_filtered['track_genre'].values

print(f"\nPCA variance explained: {pca.explained_variance_ratio_}")
print(pca_df.tail())



# Audio Features Distribution by Genre
import plotly.graph_objects as go

audio_features = ["danceability", "energy",  "loudness", "speechiness",
                  "acousticness", "instrumentalness", "liveness", "valence", "tempo"]

df_filtered_reset = df_filtered.reset_index(drop=True)

for selected_feature in ["danceability", "energy", "valence"]:
    fig = go.Figure()
    colors_gradient = ['#FF006E', '#FB5607', '#FF8C00', '#FFB703', '#FB5607']
    
    for i, genre in enumerate(sorted(df_filtered_reset['track_genre'].unique())):
        genre_data = df_filtered_reset[df_filtered_reset['track_genre'] == genre][selected_feature]
        fig.add_trace(go.Box(
            y=genre_data, 
            name=genre.upper(),
            marker_color=colors_gradient[i % len(colors_gradient)],
            boxmean='sd'
        ))
    
    fig.update_layout(
        title=f"<b>{selected_feature.capitalize()} Distribution by Genre</b>",
        yaxis_title=selected_feature.capitalize(),
        xaxis_title="Genre",
        template="plotly_dark",
        showlegend=True,
        plot_bgcolor="rgba(15, 15, 30, 0.5)",
        paper_bgcolor="rgba(15, 15, 30, 0)",
        font=dict(family="Poppins", size=12, color="#ffffff"),
        title_font_size=18,
        height=500
    )
    fig.show()

#pie chart
genre_counts = df_filtered_reset['track_genre'].value_counts().sort_values(ascending=False)
fig = px.pie(
    values=genre_counts.values,
    names=genre_counts.index,
    title="<b>Genre Distribution in Your Collection</b>",
    color_discrete_sequence=px.colors.qualitative.Bold,
    template="plotly_dark",
    hole=0.3
)
fig.update_layout(
    plot_bgcolor="rgba(15, 15, 30, 0)",
    paper_bgcolor="rgba(15, 15, 30, 0)",
    font=dict(family="Poppins", size=12, color="#ffffff"),
    title_font_size=18,
)
fig.update_traces(
    textposition='inside',
    textinfo='percent+label',
    marker=dict(line=dict(color='rgba(15, 15, 30, 0.8)', width=2))
)
fig.show()

similarity_matrix = cosine_similarity(X_pca)
df_filtered = df_filtered.reset_index(drop=True)

def recommend_song_with_filter(song_title, num_recommendation=5):
    try:
        input_idx = df_filtered[df_filtered['track_name'].str.lower() == song_title.lower()].index[0]
        input_song = df_filtered.iloc[input_idx]
    except IndexError:
        return f"Error: '{song_title}' wasn't found in this dataset. Try another track!"
    
    input_genre = str(input_song['track_genre']).lower()
    
    sim_scores = list(enumerate(similarity_matrix[input_idx]))
    

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    filtered_matches = []
    

    for index, score in sim_scores:
        if index == input_idx:
            continue

        candidate_genre = str(df_filtered.loc[index, 'track_genre']).lower()
        
        if 'indian' in input_genre and 'indian' in candidate_genre:
            filtered_matches.append(index)      
        
        elif 'indian' not in input_genre and 'indian' not in candidate_genre:
            filtered_matches.append(index)
            
        if len(filtered_matches) == num_recommendation:
            break
            
    # Look up the final row metadata and return it cleanly
    return df_filtered.iloc[filtered_matches][['track_name', 'artists', 'track_genre']]


recommend_song_with_filter("Raabta")

Dataset shape: (8000, 21)
Genres included: ['acoustic' 'black-metal' 'classical' 'dance' 'hip-hop' 'indian'
 'pop-film' 'rock']

PCA variance explained: [0.35438297 0.18104928]
           PC1      PC2 Genre
7995  1.365691 -1.22322  rock
7996  1.365691 -1.22322  rock
7997  1.365691 -1.22322  rock
7998  1.365691 -1.22322  rock
7999  1.365691 -1.22322  rock


,track_name,artists,track_genre
5078,Dil Mere,The Local Train,indian
5958,Rahguzar,Adarsh Rao,indian
5232,Khayaal,Arijit Anand;Ankita Barwad,indian
5048,Deva Deva (Malayalam),Pritam;Hesham Abdul Wahab;Arijit Singh;Arya Dh...,indian
5956,Xullo Son,Debabrata Gogoi;Rajnish Saikia,indian
